In [35]:
import pandas as pd
from pathlib import Path

REQUIRED_COLUMNS = {"Const", "Your Rating", "Title", "Year"}

In [36]:
def load_imdb_export(csv_path: str) -> pd.DataFrame:
    """
    Load a user's IMDb ratings export.
    Export from: https://www.imdb.com/list/ratings -> "..." menu -> Export
    """
    df = pd.read_csv(csv_path)

    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Unexpected IMDb export format, missing columns: {missing}")

    df = df.rename(columns={
        "Const": "imdb_id",       # e.g. tt0111161
        "Your Rating": "rating",  # 1-10 scale
        "Title": "title",
        "Year": "year",
    })

    df["rating"] = df["rating"].astype(float)
    return df[["imdb_id", "title", "year", "rating"]]

In [46]:
def map_to_movielens(user_ratings: pd.DataFrame, lookup_path: str = "models/movie_lookup.pkl") -> pd.DataFrame:
    """
    Map IMDb IDs to MovieLens movieIds using movie_lookup.pkl, which is
    indexed by movieId and has imdbId stored as a bare int (e.g. 111161
    for tt0111161 -- no 'tt' prefix, no zero-padding).
    """
    lookup = pd.read_pickle(lookup_path)

    # movieId currently lives in the index -- pull it out as a column,
    # and dedupe since the table appears to have one row per (movieId, userId)
    lookup_movies = (
        lookup.reset_index()[["movieId", "imdbId"]]
        .drop_duplicates(subset="movieId")
    )

    # user_ratings["imdb_id"] is a string like "tt0111161" -> strip to bare int
    user_ratings = user_ratings.copy()
    user_ratings["imdbId"] = (
        user_ratings["imdb_id"].str.replace("tt", "", regex=False).astype(int)
    )

    merged = user_ratings.merge(lookup_movies, on="imdbId", how="left")

    matched = merged.dropna(subset=["movieId"])
    unmatched = merged[merged["movieId"].isna()]

    if len(unmatched) > 0:
        print(f"Warning: {len(unmatched)}/{len(merged)} titles had no MovieLens match "
              f"(likely too new/obscure for the ML-20M dataset).")
        print(unmatched[["title", "year", "imdb_id"]].to_string(index=False))

    matched = matched.copy()
    matched["movieId"] = matched["movieId"].astype(int)
    return matched

In [40]:
def convert_to_10_to_5_scale(df: pd.DataFrame) -> pd.DataFrame:
    """IMDb ratings are 1-10, MovieLens ratings are 0.5-5 in 0.5 steps."""
    df = df.copy()
    df["rating"] = (df["rating"] / 2).round(1)
    return df

In [41]:
def build_new_user_profile(csv_path: str, lookup_path: str = "models/movie_lookup.pkl") -> pd.DataFrame:
    """Full pipeline: IMDb export -> MovieLens-compatible ratings dataframe."""
    raw = load_imdb_export(csv_path)
    mapped = map_to_movielens(raw, lookup_path)
    scaled = convert_to_10_to_5_scale(mapped)
    return scaled[["movieId", "rating"]]

In [53]:
build_new_user_profile("ratings.csv")

               title  year    imdb_id
              Barbie  2023  tt1517268
         Oppenheimer  2023 tt15398776
               Wonka  2023  tt6166392
          Monkey Man  2024  tt9214772
Bob Marley: One Love  2024  tt8521778
         Challengers  2024 tt16426418
             Sinners  2025 tt31193180
       Marty Supreme  2025 tt32916440
     Stranger Things  2016  tt4574334
  Brooklyn Nine-Nine  2013  tt2467372
           Community  2009  tt1439629
Arrested Development  2003  tt0367279
        Breaking Bad  2008  tt0903747
     The White Lotus  2021 tt13406094
            Southpaw  2015  tt1798684


,movieId,rating
0,4776,3.0
